# Dataset Exploration

In [ ]:
# select dataset
SUBJECT = "ants"
VERSION = "v3"

### Import Libraries

In [ ]:
from pathlib import Path
import subprocess
import warnings
warnings.filterwarnings('ignore')
%cd ../..

from PIL import Image
from IPython.display import display, Video
import cv2

import pandas as pd
import numpy as np
from datasets import Dataset, Features, Value, ClassLabel
from datasets import Image as HFImage

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
from scipy import stats
sns.set_style("darkgrid")
sns.set_palette("husl")
%matplotlib inline


## 1. Import Dataset

#### 1.1 Load annotations

In [ ]:
# Load annotations
annotations_path = Path(f"dataset/{SUBJECT}/{VERSION}/annotations.csv")

if not annotations_path.exists():
    print(f"❌ Annotations not found: {annotations_path}")
    print(f"Run: python -m src.dataset.get_annotations --subject {SUBJECT} --version {VERSION}")
else:
    df_frames = pd.read_csv(annotations_path)
    print(f"✓ Loaded {len(df_frames):,} frames from {len(df_frames['observation_id'].unique())} observations")
    print(f"\nColumns: {list(df_frames.columns)}")
    print(f"\nDataset shape: {df_frames.shape}")
    df_frames = df_frames[[col for col in df_frames.columns if not col.endswith('OL')]]
    df_frames['minute'] = (df_frames['frame_idx'] // (60 * df_frames['fps'])).astype(int)
    #df_frames = df_frames[df_frames['W_annotator'].isin(['E', 'F'])]
    display(df_frames.head())

In [ ]:
# basic statistics per frame
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

# general
print(f"\nTotal observations: {df_frames['observation_id'].nunique()}")
print(f"Total frames: {len(df_frames):,}")
print(f"Frames per observation: {len(df_frames) / df_frames['observation_id'].nunique():.0f} ± {df_frames.groupby('observation_id').size().std():.1f}")
m = df_frames.groupby("observation_id").apply(lambda g: len(g) / g["fps"].iloc[0] / 60)
total_minutes = (df_frames.groupby("observation_id").apply(lambda g: len(g) / g["fps"].iloc[0]).sum()) / 60
d, h, mi = divmod(total_minutes, 24*60)[0], *divmod(divmod(total_minutes, 24*60)[1], 60)
print(f"Minutes per observation: {m.mean():.1f} ± {m.std():.1f}")
print(f"Total duration: {total_minutes:.1f} minutes ({int(d)}d {int(h)}h {int(mi)}m)")



# treatment
print(f"\nTreatment:")
treatment_pct = (df_frames['T'].value_counts().sort_index() / len(df_frames) * 100).round(2)
obs_by_treatment = df_frames.groupby('observation_id')['T'].first().value_counts().sort_index()
for treatment, pct in treatment_pct.items():
    obs_count = int(obs_by_treatment.get(treatment, 0))
    print(f"  {treatment}: {pct}% (n={obs_count})")

# covariate 
w_cols = [c for c in df_frames.columns if c.startswith('W_')]
if w_cols:
    print(f"\nCovariates ({len(w_cols)}): {', '.join([c.replace('W_', '') for c in w_cols])}")

# outcome 
y_cols = [c for c in df_frames.columns if c.startswith('Y_')]
if y_cols:
    print(f"\nOutcomes ({len(y_cols)}): {', '.join([c.replace('Y_', '') for c in y_cols])}")
    for col in y_cols:
        print(f"\n{col}:")
        outcome_pct = (df_frames[col].value_counts().sort_index() / len(df_frames) * 100).round(2)
        for value, pct in outcome_pct.items():
            print(f"  {value}: {pct}%")


In [ ]:
# Load annotations
annotations_path = Path(f"dataset/{SUBJECT}/{VERSION}/annotations.csv")

if not annotations_path.exists():
    print(f"❌ Annotations not found: {annotations_path}")
    print(f"Run: python -m src.dataset.get_annotations --subject {SUBJECT} --version {VERSION}")
else:
    df_frames = pd.read_csv(annotations_path)
    print(f"✓ Loaded {len(df_frames):,} frames from {len(df_frames['observation_id'].unique())} observations")
    print(f"\nColumns: {list(df_frames.columns)}")
    print(f"\nDataset shape: {df_frames.shape}")
    df_frames = df_frames[[col for col in df_frames.columns if not col.endswith('OL')]]
    df_frames['minute'] = (df_frames['frame_idx'] // (60 * df_frames['fps'])).astype(int)
    df_frames = df_frames[df_frames['W_annotator'].isin(['E', 'F'])]
    display(df_frames.head())

# basic statistics per frame
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

# general
print(f"\nTotal observations: {df_frames['observation_id'].nunique()}")
print(f"Total frames: {len(df_frames):,}")
print(f"Frames per observation: {len(df_frames) / df_frames['observation_id'].nunique():.0f} ± {df_frames.groupby('observation_id').size().std():.1f}")
m = df_frames.groupby("observation_id").apply(lambda g: len(g) / g["fps"].iloc[0] / 60)
total_minutes = (df_frames.groupby("observation_id").apply(lambda g: len(g) / g["fps"].iloc[0]).sum()) / 60
d, h, mi = divmod(total_minutes, 24*60)[0], *divmod(divmod(total_minutes, 24*60)[1], 60)
print(f"Minutes per observation: {m.mean():.1f} ± {m.std():.1f}")
print(f"Total duration: {total_minutes:.1f} minutes ({int(d)}d {int(h)}h {int(mi)}m)")



# treatment
print(f"\nTreatment:")
treatment_pct = (df_frames['T'].value_counts().sort_index() / len(df_frames) * 100).round(2)
obs_by_treatment = df_frames.groupby('observation_id')['T'].first().value_counts().sort_index()
for treatment, pct in treatment_pct.items():
    obs_count = int(obs_by_treatment.get(treatment, 0))
    print(f"  {treatment}: {pct}% (n={obs_count})")

# covariate 
w_cols = [c for c in df_frames.columns if c.startswith('W_')]
if w_cols:
    print(f"\nCovariates ({len(w_cols)}): {', '.join([c.replace('W_', '') for c in w_cols])}")

# outcome 
y_cols = [c for c in df_frames.columns if c.startswith('Y_')]
if y_cols:
    print(f"\nOutcomes ({len(y_cols)}): {', '.join([c.replace('Y_', '') for c in y_cols])}")
    for col in y_cols:
        print(f"\n{col}:")
        outcome_pct = (df_frames[col].value_counts().sort_index() / len(df_frames) * 100).round(2)
        for value, pct in outcome_pct.items():
            print(f"  {value}: {pct}%")

#### 1.2 Aggregate Observations

In [ ]:
# basic statistics per observation
y_cols = [c for c in df_frames.columns if c.startswith('Y_')]
w_cols = [c for c in df_frames.columns if c.startswith('W_')]

agg_dict = {
    'n_frames': ('frame_idx', 'count'),
    'T': ('T', 'first'),
}

for col in w_cols:
    # assuming covariates do not change over observation
    agg_dict[col] = (col, 'first') 
for col in y_cols:
    agg_dict[col] = (col, 'mean')

df_obs = df_frames.groupby('observation_id').agg(**agg_dict)

display(df_obs)
# display(df_obs.describe())

In [ ]:
df_obs_T = df_obs.groupby('T')[y_cols].mean()

n_T   = df_obs.groupby('T')[y_cols].count()
std_T = df_obs.groupby('T')[y_cols].std(ddof=1)
se_T  = std_T / np.sqrt(n_T)
# Use t-distribution for confidence intervals
t_crit = n_T.applymap(lambda n: stats.t.ppf(0.975, n-1) if n > 1 else np.nan)
ci95_T = t_crit * se_T

means = df_obs_T.T                      # shape: (len(y_cols), n_treatments)
errs  = ci95_T.T.reindex_like(means)    # ensure exact same shape + labels

fig, ax = plt.subplots(figsize=(10, 6))
means.plot(
    kind='bar',
    ax=ax,
    width=0.8,
    #color=["#47E3FF", "#77DD77", "#F49AC2"],
    yerr=errs,          
    capsize=4,
)

ax.set_xlabel('Grooming')
ax.set_ylabel('Time (%)')
ax.set_xticklabels(['Blue to Focal', 'Yellow to Focal'], rotation=0, ha='right')
ax.legend(title='Treatment', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)

#display(df_obs_T)


In [ ]:
# Get unique minute values for interactive selection
minute_options = sorted(df_frames['minute'].unique())
treatments = sorted(df_frames['T'].unique())
x_tick_map = {
    'Y_B2F': 'Blue to Focal',
    'Y_Y2F': 'Yellow to Focal',
}
x_ticks = [x_tick_map.get(c, c.replace('Y_', '')) for c in y_cols]

# Create figure
fig = go.Figure()

# Extended pastel color palette for flexibility (supports up to 10 treatments)
pastel_palette = [
    '#AEC6CF',  # pastel blue
    '#FFB7B2',  # pastel red
    '#B5EAD7',  # pastel green
    '#FFD7BE',  # pastel orange
    '#E2C2FF',  # pastel purple
    '#FDFD96',  # pastel yellow
    '#C7CEEA',  # pastel lavender
    '#FFDAC1',  # pastel peach
    '#FF9AA2',  # pastel pink
    '#C1FFC1',  # pastel mint
]
treatment_colors = {
    t: pastel_palette[i % len(pastel_palette)] for i, t in enumerate(treatments)
}

# Calculate global y-axis range across all time limits
global_max = 0
for T_limit in minute_options:
    df_filtered = df_frames[df_frames['minute'] <= T_limit]
    df_obs_filtered = df_filtered.groupby('observation_id').agg(**{
        'T': ('T', 'first'),
        **{col: (col, 'mean') for col in y_cols}
    })
    df_obs_T = df_obs_filtered.groupby('T')[y_cols].mean()
    n_T = df_obs_filtered.groupby('T')[y_cols].count()
    std_T = df_obs_filtered.groupby('T')[y_cols].std(ddof=1)
    se_T = std_T / np.sqrt(n_T)
    # Use t-distribution for confidence intervals
    t_crit = n_T.applymap(lambda n: stats.t.ppf(0.975, n-1) if n > 1 else np.nan)
    ci95_T = t_crit * se_T
    
    # Track max value including error bars
    max_with_error = (df_obs_T + ci95_T).max().max()
    global_max = max(global_max, max_with_error)

# Set bar width to prevent overlap
bar_width = 0.7 / max(len(treatments), 1)

# Add traces for each time limit
for T_limit in minute_options:
    # Filter frames up to T minutes
    df_filtered = df_frames[df_frames['minute'] <= T_limit]
    df_obs_filtered = df_filtered.groupby('observation_id').agg(**{
        'T': ('T', 'first'),
        **{col: (col, 'mean') for col in y_cols}
    })
    
    # Aggregate by treatment
    df_obs_T = df_obs_filtered.groupby('T')[y_cols].mean()
    
    # Calculate errors
    n_T = df_obs_filtered.groupby('T')[y_cols].count()
    std_T = df_obs_filtered.groupby('T')[y_cols].std(ddof=1)
    se_T = std_T / np.sqrt(n_T)
    # Use t-distribution for confidence intervals
    t_crit = n_T.applymap(lambda n: stats.t.ppf(0.975, n-1) if n > 1 else np.nan)
    ci95_T = t_crit * se_T
    
    is_first = bool(T_limit == minute_options[0])
    
    # Add bars for each treatment and outcome
    for t_idx, treatment in enumerate(treatments):
        for i, outcome in enumerate(y_cols):
            mean_val = df_obs_T.loc[treatment, outcome]
            err_val = ci95_T.loc[treatment, outcome]
            x_label = x_ticks[i]
            show_legend = (i == 0)
            
            fig.add_trace(go.Bar(
                x=[x_label],
                y=[mean_val],
                name=f"T={treatment}",
                marker_color=treatment_colors.get(treatment),
                error_y=dict(type='data', array=[err_val]),
                visible=is_first,
                legendgroup=f"T={treatment}",
                showlegend=show_legend,
                offsetgroup=str(treatment),
                width=bar_width,
                hovertemplate="Outcome: %{x}<br>Treatment: %{fullData.name}<br>Mean: %{y:.2f}<extra></extra>"
            ))

# Create dropdown buttons
buttons = []
for idx, T_limit in enumerate(minute_options):
    n_treatments = len(treatments)
    n_outcomes = len(y_cols)
    start_idx = idx * n_treatments * n_outcomes
    end_idx = start_idx + n_treatments * n_outcomes
    
    visible = [False] * len(fig.data)
    visible[start_idx:end_idx] = [True] * (end_idx - start_idx)
    
    buttons.append(
        dict(
            label=f"T <= {T_limit} min",
            method='update',
            args=[{'visible': visible}]
        )
    )

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0, xanchor="left", y=1.15, yanchor="top")],
    xaxis_title="Outcome",
    yaxis_title="Time (%)",
    yaxis_range=[0, global_max * 1.05],
    barmode="group",
    bargap=0.3,
    bargroupgap=0.0,
    height=600,
    hovermode='closest',
    hoverlabel=dict(namelength=0)
 )

fig.show()


In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations

# -----------------------------
# helpers: p-value corrections
# -----------------------------
def holm_adjust(pvals: np.ndarray) -> np.ndarray:
    """Holm-Bonferroni adjusted p-values (strong FWER control)."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    out = np.empty(m, dtype=float)

    # step-down procedure
    prev = 0.0
    for k, idx in enumerate(order):
        adj = (m - k) * pvals[idx]
        adj = max(adj, prev)  # enforce monotonicity
        prev = adj
        out[idx] = min(adj, 1.0)
    return out

# -----------------------------
# helpers: blocking & shuffling
# -----------------------------
def make_block_ids(df: pd.DataFrame, block_cols):
    if not block_cols:
        return None
    # robust block key
    return df[block_cols].astype(str).agg("||".join, axis=1).to_numpy()

def permute_labels_within_blocks(t_codes: np.ndarray, block_ids, rng: np.random.Generator):
    """Return a permuted copy of t_codes, shuffling within each block."""
    if block_ids is None:
        out = t_codes.copy()
        rng.shuffle(out)
        return out

    out = t_codes.copy()
    # group indices by block once per call; okay for n~200.
    # If n is larger, precompute these lists outside.
    for b in np.unique(block_ids):
        idx = np.where(block_ids == b)[0]
        if len(idx) > 1:
            rng.shuffle(out[idx])
    return out

# -----------------------------
# core stats: one-way ANOVA F
# -----------------------------
def anova_F_from_codes(y: np.ndarray, t_codes: np.ndarray, k: int) -> float:
    """Fast one-way ANOVA F using bincount formulas."""
    n = y.size
    n_g = np.bincount(t_codes, minlength=k).astype(float)
    sum_g = np.bincount(t_codes, weights=y, minlength=k).astype(float)
    sumsq_g = np.bincount(t_codes, weights=y*y, minlength=k).astype(float)

    # guard: empty groups shouldn't happen if codes built from observed levels,
    # but if it does, return nan
    if np.any(n_g == 0):
        return np.nan

    total_sum = sum_g.sum()
    # SS_within = Σ(Σy^2 - (Σy)^2/n)
    ss_within = np.sum(sumsq_g - (sum_g * sum_g) / n_g)
    # SS_between = Σ( (Σy)^2/n ) - (Σy)^2/N
    ss_between = np.sum((sum_g * sum_g) / n_g) - (total_sum * total_sum) / n

    df_between = k - 1
    df_within = n - k
    ms_between = ss_between / df_between
    ms_within = ss_within / df_within
    return ms_between / ms_within

def permutation_anova(df: pd.DataFrame, y_col: str, t_col: str = "T",
                      block_cols=None, n_perm: int = 20000, seed: int = 0):
    """Permutation p-value for overall difference across treatments."""
    d = df[[y_col, t_col] + (block_cols or [])].dropna().copy()
    y = d[y_col].to_numpy(dtype=float)

    levels = np.sort(d[t_col].unique())
    level_to_code = {lvl: i for i, lvl in enumerate(levels)}
    t_codes = d[t_col].map(level_to_code).to_numpy(dtype=int)
    k = len(levels)

    block_ids = make_block_ids(d, block_cols)

    F_obs = anova_F_from_codes(y, t_codes, k)
    rng = np.random.default_rng(seed)

    count = 0
    for _ in range(n_perm):
        t_perm = permute_labels_within_blocks(t_codes, block_ids, rng)
        F_perm = anova_F_from_codes(y, t_perm, k)
        if F_perm >= F_obs:
            count += 1

    p = (count + 1) / (n_perm + 1)  # add-one for exactness
    return {"y_col": y_col, "levels": levels, "F_obs": F_obs, "p_perm": p, "n_perm": n_perm, "block_cols": block_cols}

# -----------------------------
# pairwise permutation test: diff in means
# -----------------------------
def diff_means_from_codes(y: np.ndarray, t_codes: np.ndarray, k: int):
    n_g = np.bincount(t_codes, minlength=k).astype(float)
    sum_g = np.bincount(t_codes, weights=y, minlength=k).astype(float)
    means = sum_g / n_g
    return means

def permutation_pairwise(df: pd.DataFrame, y_col: str, t_col: str = "T",
                         pairs=None, block_cols=None,
                         n_perm: int = 20000, seed: int = 0,
                         n_boot: int = 5000, ci_alpha: float = 0.05):
    """
    Pairwise contrasts with:
      - effect = mean(group1) - mean(group2)
      - permutation p-value (two-sided)
      - Holm-adjusted p-values
      - optional bootstrap CI (percentile) for effect
    """
    cols = [y_col, t_col] + (block_cols or [])
    d = df[cols].dropna().copy()
    y = d[y_col].to_numpy(dtype=float)

    levels = np.sort(d[t_col].unique())
    level_to_code = {lvl: i for i, lvl in enumerate(levels)}
    t_codes = d[t_col].map(level_to_code).to_numpy(dtype=int)
    k = len(levels)

    # default: all pairs
    if pairs is None:
        pairs = list(combinations(levels, 2))

    block_ids = make_block_ids(d, block_cols)
    rng = np.random.default_rng(seed)

    means_obs = diff_means_from_codes(y, t_codes, k)

    # Precompute group indices for bootstrap
    idx_by_level = {lvl: np.where(d[t_col].to_numpy() == lvl)[0] for lvl in levels}

    rows = []
    for g1, g2 in pairs:
        c1, c2 = level_to_code[g1], level_to_code[g2]
        eff_obs = means_obs[c1] - means_obs[c2]

        # permutation p-value (two-sided)
        count = 0
        for _ in range(n_perm):
            t_perm = permute_labels_within_blocks(t_codes, block_ids, rng)
            means_perm = diff_means_from_codes(y, t_perm, k)
            eff_perm = means_perm[c1] - means_perm[c2]
            if abs(eff_perm) >= abs(eff_obs):
                count += 1
        p_perm = (count + 1) / (n_perm + 1)

        # bootstrap CI for effect (stratified by treatment)
        ci_low = ci_high = np.nan
        if n_boot and n_boot > 0:
            boots = np.empty(n_boot, dtype=float)
            idx1 = idx_by_level[g1]
            idx2 = idx_by_level[g2]
            n1, n2 = len(idx1), len(idx2)
            for b in range(n_boot):
                s1 = rng.choice(idx1, size=n1, replace=True)
                s2 = rng.choice(idx2, size=n2, replace=True)
                boots[b] = y[s1].mean() - y[s2].mean()
            ci_low, ci_high = np.quantile(boots, [ci_alpha / 2, 1 - ci_alpha / 2])

        rows.append({
            "y_col": y_col,
            "g1": g1, "g2": g2,
            "effect_mean(g1-g2)": eff_obs,
            "p_perm": p_perm,
            f"ci{int((1-ci_alpha)*100)}_low": ci_low,
            f"ci{int((1-ci_alpha)*100)}_high": ci_high,
        })

    res = pd.DataFrame(rows)
    res["p_holm"] = holm_adjust(res["p_perm"].to_numpy())
    res = res.sort_values("p_perm").reset_index(drop=True)
    return res

# -----------------------------
# convenience runner
# -----------------------------
def run_inference_suite(df_obs: pd.DataFrame, outcomes=("Y_B2F", "Y_Y2F"),
                        t_col="T", block_cols=None,
                        n_perm=20000, n_boot=5000, seed=0,
                        control=None):
    """
    Runs:
      - overall permutation ANOVA for each outcome
      - pairwise tests (all pairs OR vs control if provided)
    """
    overall = []
    pairwise_tables = {}

    for y_col in outcomes:
        overall.append(permutation_anova(df_obs, y_col=y_col, t_col=t_col,
                                         block_cols=block_cols, n_perm=n_perm, seed=seed))

        levels = np.sort(df_obs[t_col].dropna().unique())
        if control is not None:
            pairs = [(lvl, control) for lvl in levels if lvl != control]
        else:
            pairs = None  # all pairs

        pairwise_tables[y_col] = permutation_pairwise(
            df_obs, y_col=y_col, t_col=t_col, pairs=pairs,
            block_cols=block_cols, n_perm=n_perm, seed=seed,
            n_boot=n_boot
        )

    return overall, pairwise_tables

In [ ]:
overall, pairwise = run_inference_suite(
    df_obs,
    outcomes=("Y_B2F", "Y_Y2F"),
    t_col="T",
    block_cols=None, #["W_batch"],  
    n_perm=20000,
    n_boot=5000,
    seed=0
)

# overall
pairwise["Y_B2F"]#.head(10)
pairwise["Y_Y2F"]#.head(20)

In [ ]:
# Get unique minute values for interactive selection
minute_options = sorted(df_frames['minute'].unique())
treatments = sorted(df_frames['T'].unique())
x_tick_map = {
    'Y_B2F': 'Blue to Focal',
    'Y_Y2F': 'Yellow to Focal',
}
x_ticks = [x_tick_map.get(c, c.replace('Y_', '')) for c in y_cols]

# Create figure
fig = go.Figure()

# Extended pastel color palette for flexibility (supports up to 10 treatments)
pastel_palette = [
    '#AEC6CF',  # pastel blue
    '#FFB7B2',  # pastel red
    '#B5EAD7',  # pastel green
    '#FFD7BE',  # pastel orange
    '#E2C2FF',  # pastel purple
    '#FDFD96',  # pastel yellow
    '#C7CEEA',  # pastel lavender
    '#FFDAC1',  # pastel peach
    '#FF9AA2',  # pastel pink
    '#C1FFC1',  # pastel mint
]
treatment_colors = {
    t: pastel_palette[i % len(pastel_palette)] for i, t in enumerate(treatments)
}

# Calculate global y-axis range across all time limits
global_max = 0
for T_limit in minute_options:
    df_filtered = df_frames[df_frames['minute'] == T_limit]
    df_obs_filtered = df_filtered.groupby('observation_id').agg(**{
        'T': ('T', 'first'),
        **{col: (col, 'mean') for col in y_cols}
    })
    df_obs_T = df_obs_filtered.groupby('T')[y_cols].mean()
    n_T = df_obs_filtered.groupby('T')[y_cols].count()
    std_T = df_obs_filtered.groupby('T')[y_cols].std(ddof=1)
    se_T = std_T / np.sqrt(n_T)
    # Use t-distribution for confidence intervals
    t_crit = n_T.applymap(lambda n: stats.t.ppf(0.975, n-1) if n > 1 else np.nan)
    ci95_T = t_crit * se_T
    
    # Track max value including error bars
    max_with_error = (df_obs_T + ci95_T).max().max()
    global_max = max(global_max, max_with_error)

# Set bar width to prevent overlap
bar_width = 0.7 / max(len(treatments), 1)

# Add traces for each time limit
for T_limit in minute_options:
    # Filter frames up to T minutes
    df_filtered = df_frames[df_frames['minute'] == T_limit]
    df_obs_filtered = df_filtered.groupby('observation_id').agg(**{
        'T': ('T', 'first'),
        **{col: (col, 'mean') for col in y_cols}
    })
    
    # Aggregate by treatment
    df_obs_T = df_obs_filtered.groupby('T')[y_cols].mean()
    
    # Calculate errors
    n_T = df_obs_filtered.groupby('T')[y_cols].count()
    std_T = df_obs_filtered.groupby('T')[y_cols].std(ddof=1)
    se_T = std_T / np.sqrt(n_T)
    # Use t-distribution for confidence intervals
    t_crit = n_T.applymap(lambda n: stats.t.ppf(0.975, n-1) if n > 1 else np.nan)
    ci95_T = t_crit * se_T
    
    is_first = bool(T_limit == minute_options[0])
    
    # Add bars for each treatment and outcome
    for t_idx, treatment in enumerate(treatments):
        for i, outcome in enumerate(y_cols):
            mean_val = df_obs_T.loc[treatment, outcome]
            err_val = ci95_T.loc[treatment, outcome]
            x_label = x_ticks[i]
            show_legend = (i == 0)
            
            fig.add_trace(go.Bar(
                x=[x_label],
                y=[mean_val],
                name=f"T={treatment}",
                marker_color=treatment_colors.get(treatment),
                error_y=dict(type='data', array=[err_val]),
                visible=is_first,
                legendgroup=f"T={treatment}",
                showlegend=show_legend,
                offsetgroup=str(treatment),
                width=bar_width,
                hovertemplate="Outcome: %{x}<br>Treatment: %{fullData.name}<br>Mean: %{y:.2f}<extra></extra>"
            ))

# Create dropdown buttons
buttons = []
for idx, T_limit in enumerate(minute_options):
    n_treatments = len(treatments)
    n_outcomes = len(y_cols)
    start_idx = idx * n_treatments * n_outcomes
    end_idx = start_idx + n_treatments * n_outcomes
    
    visible = [False] * len(fig.data)
    visible[start_idx:end_idx] = [True] * (end_idx - start_idx)
    
    buttons.append(
        dict(
            label=f"T = {T_limit} min",
            method='update',
            args=[{'visible': visible}]
        )
    )

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0, xanchor="left", y=1.15, yanchor="top")],
    xaxis_title="Outcome",
    yaxis_title="Time (%)",
    yaxis_range=[0, global_max * 1.05],
    barmode="group",
    bargap=0.3,
    bargroupgap=0.0,
    height=600,
    hovermode='closest',
    hoverlabel=dict(namelength=0)
 )

fig.show()


In [ ]:
# Total grooming per minute (average of outcomes)
minute_options = sorted(df_frames['minute'].unique())
treatments = sorted(df_frames['T'].unique())
total_label = "Total Grooming"

fig = go.Figure()

pastel_palette = [
    '#AEC6CF',  # pastel blue
    '#FFB7B2',  # pastel red
    '#B5EAD7',  # pastel green
    '#FFD7BE',  # pastel orange
    '#E2C2FF',  # pastel purple
    '#FDFD96',  # pastel yellow
    '#C7CEEA',  # pastel lavender
    '#FFDAC1',  # pastel peach
    '#FF9AA2',  # pastel pink
    '#C1FFC1',  # pastel mint
]
treatment_colors = {
    t: pastel_palette[i % len(pastel_palette)] for i, t in enumerate(treatments)
}

global_max = 0
for T_limit in minute_options:
    df_filtered = df_frames[df_frames['minute'] <= T_limit]
    df_obs_filtered = df_filtered.groupby('observation_id').agg(**{
        'T': ('T', 'first'),
        **{col: (col, 'mean') for col in y_cols}
    })
    df_obs_filtered['total_grooming'] = df_obs_filtered[y_cols].mean(axis=1)
    df_obs_T = df_obs_filtered.groupby('T')['total_grooming'].mean()
    n_T = df_obs_filtered.groupby('T')['total_grooming'].count()
    std_T = df_obs_filtered.groupby('T')['total_grooming'].std(ddof=1)
    se_T = std_T / np.sqrt(n_T)
    t_crit = n_T.apply(lambda n: stats.t.ppf(0.975, n - 1) if n > 1 else np.nan)
    ci95_T = t_crit * se_T
    max_with_error = (df_obs_T + ci95_T).max()
    global_max = max(global_max, max_with_error)

bar_width = 0.7 / max(len(treatments), 1)

for T_limit in minute_options:
    df_filtered = df_frames[df_frames['minute'] <= T_limit]
    df_obs_filtered = df_filtered.groupby('observation_id').agg(**{
        'T': ('T', 'first'),
        **{col: (col, 'mean') for col in y_cols}
    })
    df_obs_filtered['total_grooming'] = df_obs_filtered[y_cols].mean(axis=1)
    df_obs_T = df_obs_filtered.groupby('T')['total_grooming'].mean()
    n_T = df_obs_filtered.groupby('T')['total_grooming'].count()
    std_T = df_obs_filtered.groupby('T')['total_grooming'].std(ddof=1)
    se_T = std_T / np.sqrt(n_T)
    t_crit = n_T.apply(lambda n: stats.t.ppf(0.975, n - 1) if n > 1 else np.nan)
    ci95_T = t_crit * se_T
    is_first = bool(T_limit == minute_options[0])
    
    for treatment in treatments:
        mean_val = df_obs_T.loc[treatment]
        err_val = ci95_T.loc[treatment]
        fig.add_trace(go.Bar(
            x=[total_label],
            y=[mean_val],
            name=f"T={treatment}",
            marker_color=treatment_colors.get(treatment),
            error_y=dict(type='data', array=[err_val]),
            visible=is_first,
            legendgroup=f"T={treatment}",
            showlegend=True,
            offsetgroup=str(treatment),
            width=bar_width,
            hovertemplate="Outcome: %{x}<br>Treatment: %{fullData.name}<br>Mean: %{y:.2f}<extra></extra>"
        ))

buttons = []
for idx, T_limit in enumerate(minute_options):
    n_treatments = len(treatments)
    start_idx = idx * n_treatments
    end_idx = start_idx + n_treatments
    visible = [False] * len(fig.data)
    visible[start_idx:end_idx] = [True] * (end_idx - start_idx)
    buttons.append(
        dict(
            label=f"T <= {T_limit} min",
            method='update',
            args=[{'visible': visible}]
        )
    )

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0, xanchor="left", y=1.15, yanchor="top")],
    xaxis_title="Outcome",
    yaxis_title="Time (%)",
    yaxis_range=[0, global_max * 1.05],
    barmode="group",
    bargap=0.3,
    bargroupgap=0.0,
    height=600,
    hovermode='closest',
    hoverlabel=dict(namelength=0)
 )

fig.show()

In [ ]:
# minute column
df_frames['minute'] = (df_frames['frame_idx'] // (60 * df_frames['fps'])).astype(int) + 1

import ipywidgets as widgets
from IPython.display import display, clear_output

# interactive plot with selectable bin width
def plot_binned(delta_minutes: int):
    df_frames['minute_bin'] = (df_frames['minute'] // delta_minutes) * delta_minutes
    df_rep_min = (
        df_frames
        .groupby(['observation_id', 'minute_bin', 'T'], as_index=False)[y_cols]
        .mean()
    )
    xmin = int(df_rep_min['minute_bin'].min())
    xmax = int(df_rep_min['minute_bin'].max())

    fig, axes = plt.subplots(1, len(y_cols), figsize=(15, 5), sharey=True)
    for i, col in enumerate(y_cols):
        ax = axes[i]
        sns.lineplot(
            data=df_rep_min,
            x='minute_bin',
            y=col,
            hue='T',
            ax=ax,
            marker='o',
            estimator='mean',
            errorbar=('ci', 95),
        )
        ax.set_title("Blue to Focal" if col == 'Y_B2F' else "Yellow to Focal")
        ax.set_xlabel(f'Minute (binned, {delta_minutes} min)')
        ax.set_ylabel('Time (%)' if i == 0 else '')
        ax.legend(title='Treatment')
        ax.set_xlim(xmin, xmax)
    plt.tight_layout()
    plt.show()

delta_slider = widgets.IntSlider(
    value=2,
    min=1,
    max=max(2, int(df_frames['minute'].max() // 2)),
    step=1,
    description='Delta (min):',
    continuous_update=False,
    readout=True
 )

widgets.interact(plot_binned, delta_minutes=delta_slider);

## 2. Visualize

#### 2.1 Plot frames with annotations

In [ ]:
# Select random samples (stratified by treatment)
n_samples_per_treatment = 4
samples = df_frames.groupby('T').apply(lambda x: x.sample(n_samples_per_treatment, random_state=42)).reset_index(drop=True)

# Create figure
n_samples = len(samples)
n_cols = n_samples_per_treatment
n_rows = (n_samples + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten() if n_samples > 1 else [axes]

for idx, (_, row) in enumerate(samples.iterrows()):
    if idx >= len(axes):
        break
    
    # Load image (frame_path is relative from workspace root, adjust for notebook location)
    img_path = Path('dataset') / row['frame_path']
    if img_path.exists():
        img = Image.open(img_path)
        axes[idx].imshow(img)
        
        # Create title with annotations
        title = f"Obs: {row['observation_id']}, Frame: {row['frame_idx']}, T={row['T']}"
        if y_cols:
            outcomes = ", ".join([f"{col.replace('Y_', '')}={row[col]}" for col in y_cols])
            title += f"\n{outcomes}"
        
        axes[idx].set_title(title, fontsize=10)
    else:
        axes[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center')
        axes[idx].set_title(f"Missing: {img_path}", fontsize=9, color='red')
    
    axes[idx].axis('off')

# Hide unused subplots
for idx in range(n_samples, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()

# Save plot
output_dir = Path(f'outputs/{SUBJECT}/{VERSION}/frames')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'example.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Plot saved to {output_path}")

plt.show()

#### 2.1 Plot clip with annotations

In [ ]:
def create_clip(data, observation_id, start_frame=0, n_frames=30, annotate=True, output_path=None):
    """
    Create a video clip from frames with optional real-time annotations.
    
    Args:
        data: DataFrame containing frame annotations
        observation_id: Observation to extract from
        start_frame: Starting frame index (0-indexed)
        n_frames: Number of frames to include
        annotate: Whether to add annotations to frames (default: True)
        output_path: Where to save (default: temp file)
    
    Returns:
        Path to created video file
    """
    # Get frames for this observation
    obs_df_frames = data[data['observation_id'] == observation_id].sort_values('frame_idx')
    obs_df_frames = obs_df_frames.iloc[start_frame:start_frame + n_frames]
    
    if len(obs_df_frames) == 0:
        print(f"No frames found for observation {observation_id}")
        return None
    
    # Extract fps from data
    fps = obs_df_frames['fps'].iloc[0]
    
    print(f"Creating clip from {len(obs_df_frames)} frames at {fps} fps...")
    
    # Load frames
    frames = []
    frame_size = None
    
    for _, row in obs_df_frames.iterrows():
        img_path = Path('dataset') / row['frame_path']
        if img_path.exists():
            # Load with opencv (force color mode)
            img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            if img is None:
                print(f"Warning: Could not load {img_path}")
                continue
            
            # If image is grayscale, convert to BGR
            if len(img.shape) == 2:
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            
            # Set frame size from first frame
            if frame_size is None:
                frame_size = (img.shape[1], img.shape[0])  # (width, height)
            
            # Resize if needed
            if (img.shape[1], img.shape[0]) != frame_size:
                img = cv2.resize(img, frame_size)
            
            # Add annotations if requested
            if annotate:
                annotated_img = img.copy()
                
                # Set up text properties
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.5
                thickness = 1
                line_height = 20
                
                # Get treatment info
                treatment = row['T']
                
                # Create annotation text
                annotations = [
                    f"Obs: {observation_id} | T: {treatment}",
                    f"Frame: {row['frame_idx']}",
                ]
                
                # Add outcome annotations
                for col in y_cols:
                    outcome_name = col.replace('Y_', '')
                    outcome_value = row[col]
                    annotations.append(f"{outcome_name}: {outcome_value}")
                
                # Draw semi-transparent background for text
                overlay = annotated_img.copy()
                bg_height = len(annotations) * line_height + 10
                cv2.rectangle(overlay, (5, 5), (frame_size[0] - 5, bg_height), (0, 0, 0), -1)
                cv2.addWeighted(overlay, 0.6, annotated_img, 0.4, 0, annotated_img)
                
                # Draw text
                y_offset = 20
                for text in annotations:
                    cv2.putText(annotated_img, text, (10, y_offset), 
                               font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
                    y_offset += line_height
                
                frames.append(annotated_img)
            else:
                frames.append(img)
        else:
            print(f"Warning: Frame not found: {img_path}")
    
    if len(frames) == 0:
        print("No valid frames found")
        return None
    
    # Save video
    if output_path is None:
        annot_suffix = "_annotated" if annotate else ""
        output_path = f"outputs/{SUBJECT}/{VERSION}/clips/{observation_id}_{start_frame}_{n_frames}{annot_suffix}.mp4"
    
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Check if output format is GIF
    if output_path.suffix.lower() == '.gif':
        print(f"Converting {len(frames)} frames to GIF...")
        # Save frames as temporary AVI first, then convert to GIF with ffmpeg
        temp_path = str(output_path).replace('.gif', '_temp.avi')
        palette_path = str(output_path).replace('.gif', '_palette.png')
        
        try:
            print(f"Saving {len(frames)} frames to temporary file...")
            fourcc = cv2.VideoWriter_fourcc(*'MJPG')
            out = cv2.VideoWriter(temp_path, fourcc, fps, frame_size)
            
            for frame in frames:
                out.write(frame)
            out.release()
            
            # Convert to GIF with optimized color palette
            print(f"Generating optimized color palette...")
            palette_result = subprocess.run(
                ['ffmpeg', '-y', '-i', temp_path, '-vf', 'fps=30,scale=512:-1:flags=lanczos,palettegen', palette_path],
                capture_output=True, text=True
            )
            
            print(f"Converting to GIF with palette...")
            gif_result = subprocess.run(
                ['ffmpeg', '-y', '-i', temp_path, '-i', palette_path, '-filter_complex', 'fps=30,scale=512:-1:flags=lanczos[x];[x][1:v]paletteuse', str(output_path)],
                capture_output=True, text=True
            )
            
            # Cleanup
            if Path(temp_path).exists():
                Path(temp_path).unlink()
            if Path(palette_path).exists():
                Path(palette_path).unlink()
            
            if gif_result.returncode == 0:
                print(f"✓ Saved animated GIF to {output_path}")
                return output_path
            else:
                print(f"GIF optimization failed, trying simpler conversion...")
                result = subprocess.run(
                    ['ffmpeg', '-y', '-i', temp_path, '-vf', 'fps=30', str(output_path)],
                    capture_output=True, text=True
                )
                if result.returncode == 0:
                    if Path(temp_path).exists():
                        Path(temp_path).unlink()
                    print(f"✓ Saved animated GIF to {output_path}")
                    return output_path
                
        except Exception as e:
            print(f"Error: {e}")
            for p in [temp_path, palette_path]:
                if Path(p).exists():
                    Path(p).unlink()
            return None
    
    # Original MP4/AVI logic for non-GIF formats
    temp_path = str(output_path).replace('.mp4', '_temp.avi')
    
    try:
        print(f"Saving {len(frames)} frames to temporary file...")
        # Use MJPEG codec which is more universally available
        fourcc = cv2.VideoWriter_fourcc(*'MJPG')
        out = cv2.VideoWriter(temp_path, fourcc, fps, frame_size)
        
        for frame in frames:
            out.write(frame)
        out.release()
        
        # Try converting to MP4 using available encoders
        print(f"Converting to MP4...")
        
        # Try different encoder options
        encoders = [
            ['ffmpeg', '-y', '-i', temp_path, '-vcodec', 'libopenh264', '-pix_fmt', 'yuv420p', str(output_path)],
            ['ffmpeg', '-y', '-i', temp_path, '-vcodec', 'mpeg4', '-pix_fmt', 'yuv420p', str(output_path)],
        ]
        
        success = False
        for encoder_cmd in encoders:
            result = subprocess.run(encoder_cmd, capture_output=True, text=True)
            if result.returncode == 0:
                Path(temp_path).unlink()
                print(f"✓ Saved clip to {output_path}")
                success = True
                break
        
        if not success:
            # Fall back to keeping the AVI
            import shutil
            avi_path = str(output_path).replace('.mp4', '.avi')
            shutil.move(temp_path, avi_path)
            print(f"✓ Saved clip to {avi_path} (MJPEG/AVI format)")
            return avi_path
        
    except Exception as e:
        print(f"Error: {e}")
        if Path(temp_path).exists():
            Path(temp_path).unlink()
        return None
    
    return output_path

# Create a sample clip with annotations
clip_path = create_clip(data=df_frames,
                        observation_id="3_1_2", 
                        start_frame=100, 
                        n_frames=400, 
                        annotate=True)  

if clip_path:
    print(f"\n▶ Video ready for playback:")
    display(Video(str(clip_path), embed=True, width=600))

In [ ]:
# save a 5 second clip as .gif (in preview/) to be used in the README
gif_path = create_clip(data=df_frames,
                       observation_id="3_1_2",
                       start_frame=100,
                       n_frames=150,  # 5 seconds at 30 fps
                       annotate=False,
                       output_path=f"preview/{SUBJECT}/{VERSION}/demo/clip.gif")

## 3. Hugging Face dataset

In [ ]:
# Load pre-generated HF dataset (if available)
# This dataset should have been generated by: python src/dataset/get_dataset.py --subject ants --version v1
# Or by running: scripts/01_dataset/run.sh

from src.dataset.get_dataset import load_dataset

try:
    # Try to load pre-saved HF dataset from disk
    print("Attempting to load pre-generated HF dataset from disk...")
    hf_dataset = load_dataset(subject=SUBJECT, version=VERSION, from_disk=True)
    print(f"\n✓ Loaded HF dataset from disk!")
    print(f"Dataset info:\n{hf_dataset}")
    
    # Show first sample
    print(f"\n\nFirst sample:")
    sample = hf_dataset[0]
    print(f"  - observation_id: {sample['observation_id']}")
    print(f"  - frame_idx: {sample['frame_idx']}")
    print(f"  - Treatment (T): {sample['T']}")
    w_cols = [c for c in hf_dataset.column_names if c.startswith('W_')]
    if w_cols:
        print(f"  - Covariates: {', '.join([f'{c}={sample[c]}' for c in w_cols[:3]])}")
    y_cols = [c for c in hf_dataset.column_names if c.startswith('Y_')]
    if y_cols:
        print(f"  - Outcomes: {', '.join([f'{c}={sample[c]}' for c in y_cols])}")
    
    print(f"  - image shape: {sample['image'].size}")
    
except FileNotFoundError as e:
    print(f"⚠ HF dataset not found: {e}")
    print(f"\nTo generate it, run:")
    print(f"  python src/dataset/get_dataset.py --subject {SUBJECT} --version {VERSION}")
    print(f"\nOr run the full pipeline:")
    print(f"  bash scripts/01_dataset/run.sh")
except Exception as e:
    print(f"Error loading dataset: {e}")
    import traceback
    traceback.print_exc()